# MDS: Individual Subject RDMs and THINGS

This notebook projects **individual subject RDMs** and the **THINGS (adult) RDM** into 2D using **multidimensional scaling (MDS)**. Each point is one RDM; distance in the plot reflects how dissimilar the representational structures are (1 − Spearman correlation between RDM lower triangles).

**Interpretation:**
- Points close together = similar category structure (similar RDM).
- Families (subjects sharing the same family ID, e.g. first 4 digits of `subject_id`) can be colored together to see clustering by family.
- THINGS appears as a single reference point; proximity of subject RDMs to THINGS indicates how "adult-like" each subject's structure is.

**Requirements:**
- Run notebook **06** first to generate intersection-based RDMs (same categories across subjects).
- THINGS CLIP RDM from the vss-2026 pipeline (e.g. `things_clip_.../distance_matrix_alphabetical.npy` or `.csv`).

## Setup and imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.manifold import MDS
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

print("Imports OK.")

Imports OK.


## Configuration

In [2]:
# Directory containing individual subject RDMs (output of notebook 06)
SCRIPT_DIR = Path(".").resolve()
INDIVIDUAL_RDM_DIR = SCRIPT_DIR / "individual_subject_rdms_clip"
CSV_DIR = INDIVIDUAL_RDM_DIR / "csv"

# Intersection-based RDMs: same categories for all subjects (required for fair comparison).
# If no files match, the notebook auto-selects the (threshold, ncat) with the most subjects.
INTERSECTION_THRESHOLD = 28   # min number of subjects for a category to be included
INTERSECTION_NCAT = 74        # number of categories at this threshold (must match saved files)

# THINGS CLIP RDM (same embedding space as individual CLIP RDMs)
VSS_DIR = SCRIPT_DIR.parent / "vss-2026"
THINGS_THRESHOLD = "0.27"
THINGS_CLIP_RDM_CSV = VSS_DIR / f"bv_things_comp_03012026/things_clip_filtered_zscored_hierarchical_threshold_{THINGS_THRESHOLD}_163cats/distance_matrix_alphabetical.csv"
# Alternative if you have .npy + category list:
# THINGS_CLIP_RDM_NPY = same folder / "distance_matrix_alphabetical.npy"

# Output
OUTPUT_DIR = INDIVIDUAL_RDM_DIR / "mds_rdm_vs_things"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Individual RDM CSV dir: {CSV_DIR}")
print(f"THINGS RDM CSV: {THINGS_CLIP_RDM_CSV}")
print(f"Output dir: {OUTPUT_DIR}")

Individual RDM CSV dir: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/individual_analyses/individual_subject_rdms_clip/csv
THINGS RDM CSV: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/vss-2026/bv_things_comp_03012026/things_clip_filtered_zscored_hierarchical_threshold_0.27_163cats/distance_matrix_alphabetical.csv
Output dir: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/individual_analyses/individual_subject_rdms_clip/mds_rdm_vs_things


## Load intersection-based individual RDMs

In [3]:
import re

pattern = f"rdm_intersection_threshold_{INTERSECTION_THRESHOLD}_ncat{INTERSECTION_NCAT}_*.csv"
rdm_files = sorted(CSV_DIR.glob(pattern))

# If no files for requested (threshold, ncat), auto-detect from what notebook 06 saved
if not rdm_files:
    all_intersection = list(CSV_DIR.glob("rdm_intersection_threshold_*_ncat*_*.csv"))
    if not all_intersection:
        raise FileNotFoundError(
            f"No intersection RDM CSVs found in {CSV_DIR}. "
            "Run notebook 06 and generate intersection RDMs (see 'Intersection-based RDMs' section)."
        )
    # Parse (threshold, ncat) from filenames: rdm_intersection_threshold_28_ncat74_00240001.csv
    key_to_files = {}
    for path in all_intersection:
        m = re.match(r"rdm_intersection_threshold_(\d+)_ncat(\d+)_", path.name)
        if m:
            key = (int(m.group(1)), int(m.group(2)))
            key_to_files.setdefault(key, []).append(path)
    # Use (threshold, ncat) with the most files; tie-break by more categories then higher threshold
    best_key = max(key_to_files.keys(), key=lambda k: (len(key_to_files[k]), k[1], k[0]))
    rdm_files = sorted(key_to_files[best_key])
    INTERSECTION_THRESHOLD, INTERSECTION_NCAT = best_key
    print(f"Auto-selected intersection threshold={INTERSECTION_THRESHOLD}, ncat={INTERSECTION_NCAT} ({len(rdm_files)} subjects).")

subject_rdms = {}  # subject_id -> 2D array (n_cat, n_cat)
category_order = None

for path in rdm_files:
    # subject_id from filename: rdm_intersection_threshold_28_ncat74_00240001.csv -> 00240001
    subject_id = path.stem.split("_")[-1]
    df = pd.read_csv(path, index_col=0)
    rdm = df.values.astype(float)
    subject_rdms[subject_id] = rdm
    if category_order is None:
        category_order = list(df.index)

print(f"Loaded {len(subject_rdms)} subject RDMs.")
print(f"Category order (first 10): {category_order[:10]}")
print(f"RDM shape: {rdm.shape}")

Loaded 21 subject RDMs.
Category order (first 10): ['dog', 'frog', 'nose', 'face', 'hair', 'arm', 'hand', 'finger', 'toe', 'lamp']
RDM shape: (74, 74)


## Load THINGS RDM and align to same categories

In [4]:
if not THINGS_CLIP_RDM_CSV.exists():
    raise FileNotFoundError(f"THINGS RDM not found: {THINGS_CLIP_RDM_CSV}")

things_df = pd.read_csv(THINGS_CLIP_RDM_CSV, index_col=0)
things_categories_full = list(things_df.index)
things_rdm_full = things_df.values.astype(float)

# Subset THINGS to the same categories and order as intersection RDMs
cat_set = set(category_order)
missing = cat_set - set(things_categories_full)
if missing:
    raise ValueError(f"Categories in intersection not found in THINGS: {len(missing)} missing. Examples: {list(missing)[:5]}")

idx = [things_categories_full.index(c) for c in category_order]
things_rdm = things_rdm_full[np.ix_(idx, idx)]

print(f"THINGS RDM full shape: {things_rdm_full.shape}")
print(f"THINGS RDM aligned to intersection: {things_rdm.shape}")

THINGS RDM full shape: (163, 163)
THINGS RDM aligned to intersection: (74, 74)


## Build RDM-of-RDMs (dissimilarity = 1 − Spearman r)

## (Optional) MDS colored by similarity to THINGS

Points colored by dissimilarity to THINGS (red = more different from THINGS, blue = more similar / more "adult-like").

In [5]:
# Use in-memory labels, D, x, y if available; otherwise load from saved CSVs or compute from subject_rdms/things_rdm
if "labels" not in globals() or "D" not in globals() or "x" not in globals() or "y" not in globals():
    out = globals().get("OUTPUT_DIR") or Path("individual_subject_rdms_clip") / "mds_rdm_vs_things"
    coords_path = out / "mds_coordinates.csv"
    dissim_path = out / "rdm_dissimilarity_matrix.csv"
    if coords_path.exists() and dissim_path.exists():
        mds_df = pd.read_csv(coords_path)
        labels = mds_df["label"].tolist()
        x = mds_df["MDS1"].values
        y = mds_df["MDS2"].values
        dissim_df = pd.read_csv(dissim_path, index_col=0)
        D = dissim_df.values
    elif "subject_rdms" in globals() and "things_rdm" in globals():
        # Build RDM-of-RDMs and run MDS from loaded data (same logic as cells below)
        def _lower_triangle(rdm):
            n = rdm.shape[0]
            return rdm[np.tril_indices(n, k=-1)]
        def _rdm_dissimilarity(rdm_a, rdm_b):
            va, vb = _lower_triangle(rdm_a), _lower_triangle(rdm_b)
            mask = np.isfinite(va) & np.isfinite(vb)
            if mask.sum() < 10:
                return np.nan
            r, _ = spearmanr(va[mask], vb[mask])
            return np.nan if np.isnan(r) else 1.0 - r
        labels = list(subject_rdms.keys()) + ["THINGS"]
        rdm_list = [subject_rdms[sid] for sid in subject_rdms] + [things_rdm]
        n_rdms = len(labels)
        D = np.zeros((n_rdms, n_rdms))
        for i in range(n_rdms):
            for j in range(n_rdms):
                if i != j:
                    d = _rdm_dissimilarity(rdm_list[i], rdm_list[j])
                    D[i, j] = D[j, i] = d
        D_fit = D.copy()
        nan_mask = ~np.isfinite(D_fit)
        if nan_mask.any():
            fill = np.nanmax(D_fit[np.isfinite(D_fit)]) if np.any(np.isfinite(D_fit)) else 1.0
            D_fit[nan_mask] = fill
            D_fit[np.diag_indices_from(D_fit)] = 0.0
        mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42, n_init=10, max_iter=500)
        coords = mds.fit_transform(D_fit)
        x, y = coords[:, 0], coords[:, 1]
    else:
        raise NameError("Run the cells above first (Load individual RDMs, Load THINGS RDM). Or run the full notebook once to create mds_coordinates.csv and rdm_dissimilarity_matrix.csv.")
# Dissimilarity of each subject to THINGS
i_things = labels.index("THINGS")
dist_to_things = D[i_things, :].copy()
dist_to_things[i_things] = np.nan  # exclude THINGS itself for color scale

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(x[:i_things], y[:i_things], c=dist_to_things[:i_things], s=80, cmap="RdYlBu_r", 
                vmin=0, vmax=np.nanmax(dist_to_things), edgecolors="k", linewidths=0.5)
for i in range(i_things):
    ax.annotate(labels[i], (x[i], y[i]), xytext=(5, 5), textcoords="offset points", fontsize=7)
ax.scatter(x[i_things], y[i_things], c="black", s=200, marker="*", edgecolors="white", linewidths=1, label="THINGS", zorder=5)
ax.annotate("THINGS", (x[i_things], y[i_things]), xytext=(8, 8), textcoords="offset points", fontsize=11, fontweight="bold")
plt.colorbar(sc, ax=ax, label="Dissimilarity to THINGS (1 − Spearman r)")
ax.set_xlabel("MDS 1")
ax.set_ylabel("MDS 2")
ax.set_title("MDS colored by similarity to THINGS\n(blue = more similar to THINGS)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mds_colored_by_things_similarity.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {OUTPUT_DIR / 'mds_colored_by_things_similarity.png'}")

Saved: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/individual_analyses/individual_subject_rdms_clip/mds_rdm_vs_things/mds_colored_by_things_similarity.png


In [6]:
def lower_triangle(rdm):
    """Lower triangle excluding diagonal, row-major (same as np.tril_indices(..., k=-1))."""
    n = rdm.shape[0]
    return rdm[np.tril_indices(n, k=-1)]

def rdm_dissimilarity(rdm_a, rdm_b):
    """1 - Spearman r between lower triangles. 0 = identical structure, 2 = maximally dissimilar."""
    va = lower_triangle(rdm_a)
    vb = lower_triangle(rdm_b)
    mask = np.isfinite(va) & np.isfinite(vb)
    if mask.sum() < 10:
        return np.nan
    r, _ = spearmanr(va[mask], vb[mask])
    if np.isnan(r):
        return np.nan
    return 1.0 - r

# List of RDMs: all subjects then THINGS
labels = list(subject_rdms.keys()) + ["THINGS"]
rdm_list = [subject_rdms[sid] for sid in subject_rdms] + [things_rdm]
n_rdms = len(labels)

# Pairwise dissimilarity matrix
D = np.zeros((n_rdms, n_rdms))
for i in range(n_rdms):
    for j in range(n_rdms):
        if i == j:
            D[i, j] = 0.0
        else:
            d = rdm_dissimilarity(rdm_list[i], rdm_list[j])
            D[i, j] = d
            D[j, i] = d

print("RDM-of-RDMs (dissimilarity matrix) shape:", D.shape)
print("Min off-diagonal: {:.4f}, Max: {:.4f}".format(np.nanmin(D[D > 0]) if np.any(np.isfinite(D) & (D > 0)) else np.nan, np.nanmax(D)))

RDM-of-RDMs (dissimilarity matrix) shape: (22, 22)
Min off-diagonal: 0.0610, Max: 0.5294


## Run MDS

In [7]:
# Replace any remaining NaN with max observed (for MDS)
D_fit = D.copy()
nan_mask = ~np.isfinite(D_fit)
if nan_mask.any():
    fill = np.nanmax(D_fit[np.isfinite(D_fit)]) if np.any(np.isfinite(D_fit)) else 1.0
    D_fit[nan_mask] = fill
    D_fit[np.diag_indices_from(D_fit)] = 0.0

mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42, n_init=10, max_iter=500)
coords = mds.fit_transform(D_fit)

x = coords[:, 0]
y = coords[:, 1]
print(f"MDS stress: {mds.stress_:.4f}")
print(f"MDS embedding shape: {coords.shape}")

MDS stress: 0.4283
MDS embedding shape: (22, 2)


## Plot: MDS by family and THINGS

In [8]:
# Family = first 4 characters of subject_id (e.g. 0022, 0023, ...)
def get_family(sid):
    return sid[:4] if isinstance(sid, str) and sid != "THINGS" else sid

families = [get_family(lab) for lab in labels]
unique_families = sorted(set(f for f in families if f != "THINGS"))

# Colormap for families; THINGS will be plotted separately
n_fam = len(unique_families)
cmap = plt.cm.tab20 if n_fam <= 20 else plt.cm.tab20b
family_to_color = {f: cmap(i / max(n_fam, 1)) for i, f in enumerate(unique_families)}
family_to_color["THINGS"] = "black"

fig, ax = plt.subplots(figsize=(10, 8))

# Subjects
for i, lab in enumerate(labels):
    if lab == "THINGS":
        continue
    fam = families[i]
    ax.scatter(x[i], y[i], c=[family_to_color[fam]], s=80, alpha=0.9, edgecolors="k", linewidths=0.5)
    ax.annotate(lab, (x[i], y[i]), xytext=(5, 5), textcoords="offset points", fontsize=7, alpha=0.9)

# THINGS
i_things = labels.index("THINGS")
ax.scatter(x[i_things], y[i_things], c="red", s=200, marker="*", edgecolors="black", linewidths=1, label="THINGS", zorder=5)
ax.annotate("THINGS", (x[i_things], y[i_things]), xytext=(8, 8), textcoords="offset points", fontsize=11, fontweight="bold")

ax.set_xlabel("MDS 1", fontsize=12)
ax.set_ylabel("MDS 2", fontsize=12)
ax.set_title("MDS of individual RDMs and THINGS\n(dissimilarity = 1 − Spearman r; colored by family)", fontsize=12)

# Legend for families
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker="o", color="w", markerfacecolor=family_to_color[f], label=f"Family {f}", markersize=10) for f in unique_families]
legend_elements.append(Line2D([0], [0], marker="*", color="w", markerfacecolor="red", markeredgecolor="black", label="THINGS", markersize=14))
ax.legend(handles=legend_elements, loc="best", fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mds_individual_rdms_vs_things_by_family.png", dpi=150, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "mds_individual_rdms_vs_things_by_family.pdf", bbox_inches="tight")
plt.close()
print(f"Saved: {OUTPUT_DIR / 'mds_individual_rdms_vs_things_by_family.png'}")

Saved: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/individual_analyses/individual_subject_rdms_clip/mds_rdm_vs_things/mds_individual_rdms_vs_things_by_family.png


## (Optional) Save MDS coordinates and dissimilarity matrix

In [9]:
mds_df = pd.DataFrame({
    "label": labels,
    "family": families,
    "MDS1": x,
    "MDS2": y,
})
mds_df.to_csv(OUTPUT_DIR / "mds_coordinates.csv", index=False)

dissim_df = pd.DataFrame(D, index=labels, columns=labels)
dissim_df.to_csv(OUTPUT_DIR / "rdm_dissimilarity_matrix.csv")

print(f"Saved MDS coordinates to {OUTPUT_DIR / 'mds_coordinates.csv'}")
print(f"Saved RDM dissimilarity matrix to {OUTPUT_DIR / 'rdm_dissimilarity_matrix.csv'}")

Saved MDS coordinates to /home/j7yang/babyview-projects/vss2026/object-detection/analysis/individual_analyses/individual_subject_rdms_clip/mds_rdm_vs_things/mds_coordinates.csv
Saved RDM dissimilarity matrix to /home/j7yang/babyview-projects/vss2026/object-detection/analysis/individual_analyses/individual_subject_rdms_clip/mds_rdm_vs_things/rdm_dissimilarity_matrix.csv
